# JPMC HR Iceberg Hands-on Lab — Notebook 2: Features and Governance

This notebook demonstrates key Snowflake capabilities on the managed Iceberg tables created in Notebook 1:

| Section | Feature |
|---------|---------|
| **A** | Time Travel (query historical data, undrop, clone from past) |
| **B** | Warehouse Sizing (XS vs S vs M performance comparison) |
| **C** | Dynamic Data Masking (SSN and salary masking per role) |
| **D** | Hierarchical Row-Level Access (employee, manager, senior manager) |

**Prerequisite**: Run `01_Setup_and_Synthetic_Data.ipynb` first.

In [ ]:
-- Setup context
USE ROLE HOL_ADMIN;
USE DATABASE HOL_HR_ICEBERG_DB;
USE SCHEMA GOLD;
USE WAREHOUSE HOL_WH_XS;

---
## Section A: Time Travel

Snowflake supports time travel on managed Iceberg tables (up to 90 days with Enterprise edition). This lets you:
- Query data as it existed at a previous point in time
- Recover dropped tables
- Create zero-copy clones from historical snapshots

In [ ]:
-- A1: Snapshot current state — pick department 1 and check avg salary
SELECT d.department_name, d.division,
       COUNT(DISTINCT e.employee_id) AS emp_count,
       ROUND(AVG(c.base_salary), 2) AS avg_salary
FROM EMPLOYEES e
JOIN COMPENSATION c ON e.employee_id = c.employee_id AND c.effective_date = '2024-01-01'
JOIN DEPARTMENTS d ON e.department_id = d.department_id
WHERE d.department_id = 1
GROUP BY d.department_name, d.division;

In [ ]:
-- A2: Make a change — give a 15% raise to department 1 for 2024
UPDATE COMPENSATION
SET base_salary = ROUND(base_salary * 1.15, 2),
    bonus_amount = ROUND(base_salary * 1.15 * bonus_pct / 100, 2),
    total_comp = ROUND(base_salary * 1.15 + (base_salary * 1.15 * bonus_pct / 100) + (stock_units * 150), 2)
WHERE employee_id IN (SELECT employee_id FROM EMPLOYEES WHERE department_id = 1)
  AND effective_date = '2024-01-01';

-- Save the query ID for BEFORE clause
SET update_query_id = LAST_QUERY_ID();
SELECT $update_query_id AS update_query_id;

In [ ]:
-- A3: Time travel with OFFSET — see data 2 minutes ago (before the update)
SELECT d.department_name,
       ROUND(AVG(c.base_salary), 2) AS avg_salary_before
FROM COMPENSATION AT(OFFSET => -120) c
JOIN EMPLOYEES e ON c.employee_id = e.employee_id
JOIN DEPARTMENTS d ON e.department_id = d.department_id
WHERE d.department_id = 1 AND c.effective_date = '2024-01-01'
GROUP BY d.department_name;

In [ ]:
-- A4: Time travel with BEFORE(STATEMENT) — exact state before the UPDATE
SELECT d.department_name,
       ROUND(AVG(c.base_salary), 2) AS avg_salary_before_update
FROM COMPENSATION BEFORE(STATEMENT => $update_query_id) c
JOIN EMPLOYEES e ON c.employee_id = e.employee_id
JOIN DEPARTMENTS d ON e.department_id = d.department_id
WHERE d.department_id = 1 AND c.effective_date = '2024-01-01'
GROUP BY d.department_name;

In [ ]:
-- A5: UNDROP TABLE demo
CREATE OR REPLACE TABLE ATTENDANCE_BACKUP CLONE ATTENDANCE;

SELECT 'Before drop', COUNT(*) FROM ATTENDANCE_BACKUP;

DROP TABLE ATTENDANCE_BACKUP;
-- Table is gone:
-- SELECT COUNT(*) FROM ATTENDANCE_BACKUP; -- This would fail

UNDROP TABLE ATTENDANCE_BACKUP;
SELECT 'After undrop', COUNT(*) FROM ATTENDANCE_BACKUP;

DROP TABLE ATTENDANCE_BACKUP; -- cleanup

In [ ]:
-- A6: Zero-copy clone from a past point in time
CREATE OR REPLACE TABLE COMPENSATION_SNAPSHOT
  CLONE COMPENSATION BEFORE(STATEMENT => $update_query_id);

-- Verify: snapshot has the pre-raise salary
SELECT 'Snapshot (pre-raise)' AS label,
       ROUND(AVG(base_salary), 2) AS avg_salary
FROM COMPENSATION_SNAPSHOT
WHERE employee_id IN (SELECT employee_id FROM EMPLOYEES WHERE department_id = 1)
  AND effective_date = '2024-01-01'
UNION ALL
SELECT 'Current (post-raise)',
       ROUND(AVG(base_salary), 2)
FROM COMPENSATION
WHERE employee_id IN (SELECT employee_id FROM EMPLOYEES WHERE department_id = 1)
  AND effective_date = '2024-01-01';

DROP TABLE COMPENSATION_SNAPSHOT; -- cleanup

---
## Section B: Warehouse Sizing

We run the same analytical query on 3 warehouse sizes to show the cost-vs-speed tradeoff.

| Warehouse | Size | Credits/Hour |
|-----------|------|-------------|
| HOL_WH_XS | X-Small | 1 |
| HOL_WH_S | Small | 2 |
| HOL_WH_M | Medium | 4 |

In [ ]:
-- B1: Benchmark query — complex analytical over 900K comp + 300K employees
-- Window functions, multi-join, aggregations with HAVING

-- Run on X-Small
USE WAREHOUSE HOL_WH_XS;

SELECT d.division, d.department_name,
       e.job_level,
       COUNT(DISTINCT e.employee_id) AS headcount,
       ROUND(AVG(c.total_comp), 2)   AS avg_total_comp,
       ROUND(MEDIAN(c.total_comp), 2) AS median_total_comp,
       ROUND(STDDEV(c.total_comp), 2) AS stddev_comp,
       ROUND(AVG(pr.overall_rating), 2) AS avg_rating,
       SUM(c.stock_units) AS total_stock_units,
       RANK() OVER (PARTITION BY d.division ORDER BY AVG(c.total_comp) DESC) AS comp_rank_in_division,
       ROUND(AVG(c.total_comp) - AVG(AVG(c.total_comp)) OVER (PARTITION BY d.division), 2) AS comp_vs_division_avg
FROM EMPLOYEES e
JOIN COMPENSATION c ON e.employee_id = c.employee_id AND c.effective_date = '2024-01-01'
JOIN DEPARTMENTS d ON e.department_id = d.department_id
JOIN PERFORMANCE_REVIEWS pr ON e.employee_id = pr.employee_id AND pr.review_period = 'H1-2024'
WHERE e.employment_status = 'Active'
GROUP BY d.division, d.department_name, e.job_level
HAVING COUNT(DISTINCT e.employee_id) > 10
ORDER BY d.division, avg_total_comp DESC;

SET qid_xs = LAST_QUERY_ID();
SELECT $qid_xs AS xs_query_id;

In [ ]:
-- B2: Run on Small
USE WAREHOUSE HOL_WH_S;

SELECT d.division, d.department_name,
       e.job_level,
       COUNT(DISTINCT e.employee_id) AS headcount,
       ROUND(AVG(c.total_comp), 2)   AS avg_total_comp,
       ROUND(MEDIAN(c.total_comp), 2) AS median_total_comp,
       ROUND(STDDEV(c.total_comp), 2) AS stddev_comp,
       ROUND(AVG(pr.overall_rating), 2) AS avg_rating,
       SUM(c.stock_units) AS total_stock_units,
       RANK() OVER (PARTITION BY d.division ORDER BY AVG(c.total_comp) DESC) AS comp_rank_in_division,
       ROUND(AVG(c.total_comp) - AVG(AVG(c.total_comp)) OVER (PARTITION BY d.division), 2) AS comp_vs_division_avg
FROM EMPLOYEES e
JOIN COMPENSATION c ON e.employee_id = c.employee_id AND c.effective_date = '2024-01-01'
JOIN DEPARTMENTS d ON e.department_id = d.department_id
JOIN PERFORMANCE_REVIEWS pr ON e.employee_id = pr.employee_id AND pr.review_period = 'H1-2024'
WHERE e.employment_status = 'Active'
GROUP BY d.division, d.department_name, e.job_level
HAVING COUNT(DISTINCT e.employee_id) > 10
ORDER BY d.division, avg_total_comp DESC;

SET qid_s = LAST_QUERY_ID();
SELECT $qid_s AS s_query_id;

In [ ]:
-- B3: Run on Medium
USE WAREHOUSE HOL_WH_M;

SELECT d.division, d.department_name,
       e.job_level,
       COUNT(DISTINCT e.employee_id) AS headcount,
       ROUND(AVG(c.total_comp), 2)   AS avg_total_comp,
       ROUND(MEDIAN(c.total_comp), 2) AS median_total_comp,
       ROUND(STDDEV(c.total_comp), 2) AS stddev_comp,
       ROUND(AVG(pr.overall_rating), 2) AS avg_rating,
       SUM(c.stock_units) AS total_stock_units,
       RANK() OVER (PARTITION BY d.division ORDER BY AVG(c.total_comp) DESC) AS comp_rank_in_division,
       ROUND(AVG(c.total_comp) - AVG(AVG(c.total_comp)) OVER (PARTITION BY d.division), 2) AS comp_vs_division_avg
FROM EMPLOYEES e
JOIN COMPENSATION c ON e.employee_id = c.employee_id AND c.effective_date = '2024-01-01'
JOIN DEPARTMENTS d ON e.department_id = d.department_id
JOIN PERFORMANCE_REVIEWS pr ON e.employee_id = pr.employee_id AND pr.review_period = 'H1-2024'
WHERE e.employment_status = 'Active'
GROUP BY d.division, d.department_name, e.job_level
HAVING COUNT(DISTINCT e.employee_id) > 10
ORDER BY d.division, avg_total_comp DESC;

SET qid_m = LAST_QUERY_ID();
SELECT $qid_m AS m_query_id;

In [ ]:
-- B4: Compare execution times
SELECT query_id, warehouse_name, warehouse_size,
       execution_status,
       total_elapsed_time / 1000 AS total_elapsed_sec,
       compilation_time,
       execution_time,
       bytes_scanned,
       rows_produced,
       partitions_scanned,
       partitions_total,
       bytes_spilled_to_local_storage,
       bytes_spilled_to_remote_storage,
       percentage_scanned_from_cache
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
WHERE query_id IN ($qid_xs, $qid_s, $qid_m)
ORDER BY warehouse_size;

### Interpretation
- **X-Small**: Cheapest (1 credit/hr) but slowest. May spill to disk on large joins.
- **Small**: 2x the compute. Typically cuts execution time significantly for data that doesn't fit in XS memory.
- **Medium**: 4x the compute. Diminishing returns if the data set fits in Small's memory.

**Key takeaway**: Right-size your warehouse for the workload. Pay for speed only when needed.

---
## Section C: Dynamic Data Masking

Masking policies let you control what data each role sees, without duplicating tables or creating views. The policy is applied at the column level and enforced transparently at query time.

In [ ]:
-- C1: SSN Masking Policy
USE ROLE HOL_ADMIN;
USE SCHEMA HOL_HR_ICEBERG_DB.GOVERNANCE;

CREATE OR REPLACE MASKING POLICY SSN_MASK AS (val STRING) RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() = 'HOL_ADMIN'      THEN val                            -- Full SSN
    WHEN CURRENT_ROLE() = 'HOL_HR_ANALYST'  THEN 'XXX-XX-' || RIGHT(val, 4)    -- Last 4 digits
    ELSE '***-**-****'                                                          -- Fully masked
  END;

-- C2: Salary Masking Policy
CREATE OR REPLACE MASKING POLICY SALARY_MASK AS (val NUMBER(12,2)) RETURNS NUMBER(12,2) ->
  CASE
    WHEN CURRENT_ROLE() IN ('HOL_ADMIN', 'HOL_HR_ANALYST', 'HOL_MANAGER', 'HOL_SENIOR_MANAGER')
      THEN val
    ELSE NULL   -- HOL_EMPLOYEE sees NULL
  END;

In [ ]:
-- C3: Apply masking policies to columns
USE SCHEMA HOL_HR_ICEBERG_DB.GOLD;

ALTER ICEBERG TABLE EMPLOYEES MODIFY COLUMN ssn
  SET MASKING POLICY HOL_HR_ICEBERG_DB.GOVERNANCE.SSN_MASK;

ALTER ICEBERG TABLE COMPENSATION MODIFY COLUMN base_salary
  SET MASKING POLICY HOL_HR_ICEBERG_DB.GOVERNANCE.SALARY_MASK;

ALTER ICEBERG TABLE COMPENSATION MODIFY COLUMN total_comp
  SET MASKING POLICY HOL_HR_ICEBERG_DB.GOVERNANCE.SALARY_MASK;

In [ ]:
-- C4: Test as HOL_ADMIN — sees full SSN, full salary
USE ROLE HOL_ADMIN;
SELECT employee_id, first_name, last_name, ssn, job_title
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES LIMIT 5;

SELECT employee_id, base_salary, total_comp
FROM HOL_HR_ICEBERG_DB.GOLD.COMPENSATION LIMIT 5;

In [ ]:
-- C5: Test as HOL_HR_ANALYST — sees last 4 of SSN, full salary
USE ROLE HOL_HR_ANALYST;
USE WAREHOUSE HOL_WH_XS;

SELECT employee_id, first_name, last_name, ssn, job_title
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES LIMIT 5;

SELECT employee_id, base_salary, total_comp
FROM HOL_HR_ICEBERG_DB.GOLD.COMPENSATION LIMIT 5;

In [ ]:
-- C6: Test as HOL_EMPLOYEE — sees fully masked SSN, NULL salary
USE ROLE HOL_EMPLOYEE;
USE WAREHOUSE HOL_WH_XS;

SELECT employee_id, first_name, last_name, ssn, job_title
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES LIMIT 5;

SELECT employee_id, base_salary, total_comp
FROM HOL_HR_ICEBERG_DB.GOLD.COMPENSATION LIMIT 5;

---
## Section D: Hierarchical Row-Level Access Policies

Row access policies restrict which **rows** a role can see. We implement a 3-tier hierarchy:

| Role | Sees |
|------|------|
| `HOL_ADMIN` / `HOL_HR_ANALYST` | All 300K employees |
| `HOL_SENIOR_MANAGER` | All employees in their department (~1,500) |
| `HOL_MANAGER` | Direct reports + self (~5-7 rows) |
| `HOL_EMPLOYEE` | Only their own row (1 row) |

The policy uses the `GOVERNANCE.ROLE_ACCESS_MAPPING` table to look up which employee_id and department_id the current role is mapped to.

In [ ]:
-- D1: Create Row Access Policy
USE ROLE HOL_ADMIN;
USE SCHEMA HOL_HR_ICEBERG_DB.GOVERNANCE;

CREATE OR REPLACE ROW ACCESS POLICY HR_ROW_ACCESS
  AS (row_employee_id NUMBER, row_department_id NUMBER, row_manager_id NUMBER)
  RETURNS BOOLEAN ->
    -- Admins and analysts see everything
    CURRENT_ROLE() IN ('HOL_ADMIN', 'HOL_HR_ANALYST', 'ACCOUNTADMIN')
    -- Senior Manager: sees all employees in their department
    OR (CURRENT_ROLE() = 'HOL_SENIOR_MANAGER'
        AND row_department_id IN (
            SELECT department_id FROM HOL_HR_ICEBERG_DB.GOVERNANCE.ROLE_ACCESS_MAPPING
            WHERE role_name = 'HOL_SENIOR_MANAGER'))
    -- Manager: sees direct reports (where row's manager_id = the mapped employee) + own row
    OR (CURRENT_ROLE() = 'HOL_MANAGER'
        AND (row_manager_id IN (
                SELECT employee_id FROM HOL_HR_ICEBERG_DB.GOVERNANCE.ROLE_ACCESS_MAPPING
                WHERE role_name = 'HOL_MANAGER')
             OR row_employee_id IN (
                SELECT employee_id FROM HOL_HR_ICEBERG_DB.GOVERNANCE.ROLE_ACCESS_MAPPING
                WHERE role_name = 'HOL_MANAGER')))
    -- Employee: sees only their own row
    OR (CURRENT_ROLE() = 'HOL_EMPLOYEE'
        AND row_employee_id IN (
            SELECT employee_id FROM HOL_HR_ICEBERG_DB.GOVERNANCE.ROLE_ACCESS_MAPPING
            WHERE role_name = 'HOL_EMPLOYEE'));

In [ ]:
-- D2: Apply Row Access Policy to tables
USE SCHEMA HOL_HR_ICEBERG_DB.GOLD;

-- EMPLOYEES: direct mapping
ALTER ICEBERG TABLE EMPLOYEES ADD ROW ACCESS POLICY
  HOL_HR_ICEBERG_DB.GOVERNANCE.HR_ROW_ACCESS
  ON (employee_id, department_id, manager_id);

-- COMPENSATION: join on employee_id, use 0 for dept/manager (policy checks employee_id)
ALTER ICEBERG TABLE COMPENSATION ADD ROW ACCESS POLICY
  HOL_HR_ICEBERG_DB.GOVERNANCE.HR_ROW_ACCESS
  ON (employee_id, employee_id, employee_id);

-- PERFORMANCE_REVIEWS: same pattern
ALTER ICEBERG TABLE PERFORMANCE_REVIEWS ADD ROW ACCESS POLICY
  HOL_HR_ICEBERG_DB.GOVERNANCE.HR_ROW_ACCESS
  ON (employee_id, employee_id, employee_id);

-- ATTENDANCE: same pattern
ALTER ICEBERG TABLE ATTENDANCE ADD ROW ACCESS POLICY
  HOL_HR_ICEBERG_DB.GOVERNANCE.HR_ROW_ACCESS
  ON (employee_id, employee_id, employee_id);

In [ ]:
-- D3: Test as HOL_EMPLOYEE — should see only 1 row (their own)
USE ROLE HOL_EMPLOYEE;
USE WAREHOUSE HOL_WH_XS;

SELECT 'HOL_EMPLOYEE' AS role, COUNT(*) AS visible_rows
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES;

SELECT employee_id, first_name, last_name, ssn, job_title, department_id
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES;

In [ ]:
-- D4: Test as HOL_MANAGER — should see direct reports + self (~5-7 rows)
USE ROLE HOL_MANAGER;
USE WAREHOUSE HOL_WH_XS;

SELECT 'HOL_MANAGER' AS role, COUNT(*) AS visible_rows
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES;

SELECT employee_id, first_name, last_name, job_title, job_level, department_id, manager_id
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES
ORDER BY job_level DESC;

In [ ]:
-- D5: Test as HOL_SENIOR_MANAGER — should see entire department (~1,500 rows)
USE ROLE HOL_SENIOR_MANAGER;
USE WAREHOUSE HOL_WH_XS;

SELECT 'HOL_SENIOR_MANAGER' AS role, COUNT(*) AS visible_rows
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES;

SELECT job_level, COUNT(*) AS cnt
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES
GROUP BY job_level ORDER BY job_level;

In [ ]:
-- D6: Test as HOL_ADMIN — should see all 300K rows
USE ROLE HOL_ADMIN;

SELECT 'HOL_ADMIN' AS role, COUNT(*) AS visible_rows
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES;

In [ ]:
-- D7: Combined demo — masking + row access together
-- As HOL_EMPLOYEE: 1 row, masked SSN, NULL salary
USE ROLE HOL_EMPLOYEE;
USE WAREHOUSE HOL_WH_XS;

SELECT e.employee_id, e.first_name, e.last_name, e.ssn, e.job_title,
       c.base_salary, c.total_comp
FROM HOL_HR_ICEBERG_DB.GOLD.EMPLOYEES e
LEFT JOIN HOL_HR_ICEBERG_DB.GOLD.COMPENSATION c
  ON e.employee_id = c.employee_id AND c.effective_date = '2024-01-01';

In [ ]:
-- D8: Optional Cleanup
-- Uncomment to remove all lab objects
-- USE ROLE ACCOUNTADMIN;
-- DROP DATABASE IF EXISTS HOL_HR_ICEBERG_DB;
-- DROP WAREHOUSE IF EXISTS HOL_WH_XS;
-- DROP WAREHOUSE IF EXISTS HOL_WH_S;
-- DROP WAREHOUSE IF EXISTS HOL_WH_M;
-- DROP ROLE IF EXISTS HOL_EMPLOYEE;
-- DROP ROLE IF EXISTS HOL_MANAGER;
-- DROP ROLE IF EXISTS HOL_SENIOR_MANAGER;
-- DROP ROLE IF EXISTS HOL_HR_ANALYST;
-- DROP ROLE IF EXISTS HOL_ADMIN;

---
## Summary

| Feature | What We Demonstrated |
|---------|---------------------|
| **Time Travel** | Query historical data with AT/BEFORE, UNDROP, zero-copy clone from past |
| **Warehouse Sizing** | Same query on XS/S/M — cost vs speed tradeoff |
| **Dynamic Masking** | SSN masked per role (full / last-4 / hidden), salary hidden for employees |
| **Row Access** | 3-tier hierarchy: employee (1 row), manager (~5-7), senior manager (~1,500), admin (300K) |

All features work transparently on **managed Iceberg tables** — no application changes needed.